### Build a Local RAG System with Open-Source LLMs

Model used: `deepseek-r1:1.5b `

|Step 1: Export current packages from Python 3.10
pip freeze > requirements.txt

In [1]:
# Import al ncessary libraries
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Get chat response
from ollama import chat

response = chat(
    model='llama3.2:3b',    
    messages=[
        {'role': 'user', 'content': 'Hello! What model are you? And what are you specially good at?'}
        ],
)
print(response.message.content)

Nice to meet you! I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI." My primary function is to process and generate human-like text based on the input I receive.

I'm particularly good at:

1. **Answering questions**: I can provide information on a wide range of topics, from science and history to entertainment and culture.
2. **Generating text**: I can create coherent and context-specific text on various subjects, including writing articles, stories, or even entire conversations like this one.
3. **Translation**: I can translate text from one language to another, helping bridge the communication gap between people who speak different languages.
4. **Text summarization**: I can summarize long pieces of text into concise and informative summaries, making it easier to grasp key points.
5. **Creative writing**: I can use my imagination to generate creative content, such as poetry or short stories.

Keep in mind that while I'm incredibly h

Step 1: Load the PDF Document by using langchain library.

PyPDFDirectory reads only files that are PDFs. It will ignore non-PDFs.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser
import os

def load_documents_from_directory(directory_path):
    all_docs = []
    # Loop through directory to find PDFs
    for filename in os.listdir(directory_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(directory_path, filename)
            # PyMuPDF is fast and supports image extraction for OCR
            loader = PyMuPDFLoader(
                file_path, 
                extract_images=True, 
                images_parser=TesseractBlobParser() # Ensure tesseract is installed on your system
            )
            all_docs.extend(loader.load())
    return all_docs
directory_path = "C:/Users/user/OneDrive/Documents/Own Projects/RAG/Data Files"
documents = load_documents_from_directory(directory_path)

Documents are too big, hence the documents variable len is alot for 3 pdf files.

In [4]:
print(documents[0])

page_content='ONG Jun Kye 
Hong Kong 
https://www.linkedin.com/in/ojunkye/ | https://www.github.com/EOngjk 
LANGUAGES 
• 
English (Fluent), Mandarin (Conversational), Cantonese (Conversational), Malay (Fluent) 
 
EDUCATION 
Hong Kong Baptist University (HKBU)                                                                                                                                              Kowloon Tong, Hong Kong 
BSc (Hons) in Business Computing and Data Analytics (BCDA) 
     
  2021 – 2025 
• 
cGPA: 3.95 / 4.0 
• 
Coursework: Financial Management; Investment Management; Management Information Systems; Problem-Solving Using Object Oriented 
Approach in Java; Data Structures and Algorithms; Database Management; Business Intelligence and Decision Support; Big Data Analytics 
with Python; Financial Technology for Banking and Finance; Exploratory Data Analysis and Visualization; Data Mining and Knowledge 
Discovery, Natural Language Processing, Advanced Business Analytics and Dat

Step 2: Split Documents into Chunks

Function split_documents_into_chunks() takes in a list of Document type as input and splits the documents into smaller chunks.

- Text splitter, split based on character counts (chunk size).
    - Chunk size = 1000 meaning size of each chunk is 500 (can go over if cant find good natural break point)
    - Chunk overlap = 200 means an overlap of 50 characters between consecutive chunks. Helps maintain context accross chunks which is useful for contextual understanding of the text.
- split_document: takes loaded docs and applies splitting logic to each one, and returns a list called chunks where each item is a smaller text segment (a "chunk") derived from original doc.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_documents_into_chunks(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False
    )
    return text_splitter.split_documents(documents)


In [6]:
# documents = load_documents_from_directory(directory_path)
chunks = split_documents_into_chunks(documents)
print(f"Number of chunks: {len(chunks)}")
print(chunks[1])

Number of chunks: 87
page_content='2021 – 2025 
• 
cGPA: 3.95 / 4.0 
• 
Coursework: Financial Management; Investment Management; Management Information Systems; Problem-Solving Using Object Oriented 
Approach in Java; Data Structures and Algorithms; Database Management; Business Intelligence and Decision Support; Big Data Analytics 
with Python; Financial Technology for Banking and Finance; Exploratory Data Analysis and Visualization; Data Mining and Knowledge' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-06T17:50:48+08:00', 'source': 'C:/Users/user/OneDrive/Documents/Own Projects/RAG/Data Files\\Eric_CV.pdf', 'file_path': 'C:/Users/user/OneDrive/Documents/Own Projects/RAG/Data Files\\Eric_CV.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Eric Ong', 'subject': '', 'keywords': '', 'moddate': '2026-04-06T17:50:48+08:00', 'trapped': '', 'modDate': "D:20260406175048+08'00'", 'creati

Step 3: Create Embeddings

To search through our text chunks, we need to turn them into vector embeddings.

Need to create an function that calls an embedding function, as we are using it at 2 different places:
1. When we create database
2. When we want to query the database.

Important to use the exact same embedding functions.

Here we are using HuggingFaceEmbeddings with the BAAI/bge-small-en model. It’s free, lightweight (max 512 tokens), fast, and works completely offline, which is great for local setups. 

Else, want bigger chunk sizes for more context, can use AWS' BedrockEmbeddings `from langchain_community,embeddings.bedrock import BedrockEmbeddings`, bigger Maximum Token of 8,192 tokens.

In [7]:
# Import embedding model for sentence embeddings
from langchain_huggingface import HuggingFaceEmbeddings

def get_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en"
    )
    return embeddings

Step 4: Create the Vector Database

Save the embeddings in Chroma, an open-source vector database.

Setting a persist_directory to save the database to my own computer. 

This way I won’t need to process the PDF again each time I run the script.

In [8]:
from langchain_chroma import Chroma
from torch import chunk
import shutil
import gc
import time

chroma_path = "./local_chroma_db"

def calculate_chunk_ids(chunks):
    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"

        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        chunk_id = f"{current_page_id}_{current_chunk_index}"
        last_page_id = current_page_id
        chunk.metadata["id"] = chunk_id

    return chunks


def add_to_chroma(chunks):
    chunks.sort(key=lambda x: (x.metadata.get("source", ""), x.metadata.get("page", 0)))
    db = None

    try:
        db = Chroma(
            persist_directory=chroma_path,
            embedding_function=get_embeddings()
        )

        chunks_with_ids = calculate_chunk_ids(chunks)
        existing_items = db.get(include=[])
        existing_ids = set(existing_items["ids"])
        print(f"Number of existing documents in DB: {len(existing_ids)}")

        new_chunks = [chunk for chunk in chunks_with_ids if chunk.metadata["id"] not in existing_ids]

        if new_chunks:
            print(f"Adding new documents: {len(new_chunks)}")
            new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
            db.add_documents(new_chunks, ids=new_chunk_ids)
        else:
            print("No new documents to add")

    finally:
        if db is not None:
            del db
        gc.collect()
        time.sleep(1)


def clear_database():
    gc.collect()
    time.sleep(1)

    if os.path.exists(chroma_path):
        print("🗑️ Clearing database...")

        for attempt in range(5):
            try:
                shutil.rmtree(chroma_path)
                print("✅ Database cleared.")
                break
            except PermissionError as e:
                print(f"⚠️ Attempt {attempt + 1}: DB still in use: {e}")
                gc.collect()
                time.sleep(2)
        else:
            print("❌ Failed to delete database after multiple retries.")
            print("Tip: Restart kernel / close notebook and try again.")


In [9]:
clear_database()

🗑️ Clearing database...
⚠️ Attempt 1: DB still in use: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
⚠️ Attempt 2: DB still in use: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
⚠️ Attempt 3: DB still in use: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
⚠️ Attempt 4: DB still in use: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
⚠️ Attempt 5: DB still in use: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
❌ Failed to delete database after multiple retries.
Tip: Restart kernel / close notebook and try again.


Step 5: Set Up the Retriever & Reranker

Retriever acts as the search engine for the RAG System.
- this is when we directly return chroma db and search from there.

Reranker is an AI model that acts as a "second-pass" filter to improve the accuracy of search results.
- In a standard search or RAG pipeline, it re-evaluates a shortlist of candidates and re-orders them to ensure the absolute best matches are at the very top. 


In [11]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

def get_hybrid_retriever(chunks, db: Chroma):
    # 1. Semantic Retriever (Your current Chroma setup)
    vector_retriever = db.as_retriever(search_kwargs={"k": 10}) # Fetch more for reranking

    # 2. Keyword Retriever (BM25)
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 10

    # 3. Combine both (Weights: 0.5/0.5 is a good start)
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.5, 0.5]
    )

    # 4. Reranker to boost relevance
    # Cross-Encoder "reads" content to fix the section-bleeding issue.
    compressor = FlashrankRerank(top_n=3)
    
    final_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, 
        base_retriever=ensemble_retriever
    )
    
    return final_retriever

Step 6: Connect to the Local LLM.

Imports Ollama adapter class from langchain_ollama (LangChain wrapper for Ollama API/local server).

In [12]:
from langchain_ollama import OllamaLLM

# This constructs a LLM object that can be used to generate responses from the Ollama model.
llm = OllamaLLM(
    model="llama3.2:3b"
)

Step 7: Define the Prompt Template

Create a prompt that instructs the LLM to use only the retrieved context to answer the question and to keep the answer short, no more than three sentences.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert AI Assistant specializing in the analysing provided documents and answering questions based on the retrieved context.

### CONTEXT:
{context}

### QUESTION:
{question}

### INSTRUCTIONS:
1. USE ONLY the provided 'Context' to answer the 'Question'. 
2. IF the answer is not contained within the Context, explicitly state: "I do not have enough information in the provided documents to answer this." 
3. DO NOT use outside knowledge or make up facts.
4. Answer concisely and use a professional tone.

### HELPFUL ANSWER:
"""
)

Step 8: Build the query rag

We use LangChain Expression Language (LCEL) to connect all the parts. The pipeline takes your question, finds the documents, puts them into one string, sends the prompt to the LLM, and then gives you a clean answer.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def query_rag(query_text: str):
    db = None
    retriever = None
    rag_chain = None

    try:
        embedding_function = get_embeddings()
        db = Chroma(
            persist_directory=chroma_path,
            embedding_function=embedding_function
        )

        retriever = get_hybrid_retriever(chunks, db)

        rag_chain = (
            {
                "context": retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        response_text = rag_chain.invoke(query_text)

        relevant_docs = retriever.invoke(query_text)
        sources = [doc.metadata.get("id", "Unknown") for doc in relevant_docs]

        formatted_response = f"Question: {query_text}\nResponse: {response_text}\nSources: {sources}"
        print(formatted_response)

        return response_text

    finally:
        if db is not None:
            del db
        gc.collect()
        time.sleep(1)

In [ ]:
# Clear database on each run to avoid duplicates during development. Comment out in production.
clear_database()
print("\nLocal RAG System Ready!")
print("Type 'exit' to quit\n")

# Create (or update) the data store.
# documents = load_documents_from_directory(directory_path)
chunks = split_documents_into_chunks(documents)
add_to_chroma(chunks)

while True:
    question = input("Ask a question: ")

    if question.lower() == "exit":  
        break

    response = query_rag(question)
    print("\n")

🗑️ Clearing database...
⚠️ Could not delete DB because it's in use. Waiting 1 second...
❌ Failed to delete: [WinError 5] Access is denied: './local_chroma_db\\a5cf998c-c662-4840-8134-62ff4c9c96c2'
Tip: Close any other Python processes or VS Code windows using this DB.

Local RAG System Ready!
Type 'exit' to quit



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of existing documents in DB: 137
No new documents to add


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Question: what about his work experience?
Response: Based on the provided context, I do not have enough information in the documents to answer the question about Eric's work experience. The text only mentions Eric's performance and skills, but does not provide any details about his previous employment or work history.
Sources: ['C:\\Users\\user\\OneDrive\\Documents\\Own Projects\\RAG\\Data Files\\OngJunKye_RefLetter.pdf:1_1', 'C:\\Users\\user\\OneDrive\\Documents\\Own Projects\\RAG\\Data Files\\OngJunKye_RefLetter.pdf:2_0', 'C:/Users/user/OneDrive/Documents/Own Projects/RAG/Data Files\\OngJunKye_RefLetter.pdf:1_2']




INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en/tree/main

Question: 
Response: I'd be happy to help you with your question based on the provided context. Please go ahead and ask your question, and I'll do my best to provide a concise answer using only the information from the context.
Sources: ['C:\\Users\\user\\OneDrive\\Documents\\Own Projects\\RAG\\Data Files\\monopoly.pdf:5_0', 'C:/Users/user/OneDrive/Documents/Own Projects/RAG/Data Files\\monopoly.pdf:1_2', 'C:\\Users\\user\\OneDrive\\Documents\\Own Projects\\RAG\\Data Files\\monopoly.pdf:1_0']


